# Insurance FWA Risk Scoring — EDA & Modeling Notebook

**Project**: Insurance Fraud, Waste & Abuse (FWA) Risk Scoring & GenAI Claims Review System  
**Author**: Portfolio demonstration project  
**Data**: Synthetic, generated via `src/data_generation.py`  

This notebook walks through:
1. Data loading and exploration
2. Exploratory data analysis (EDA)
3. Feature engineering overview
4. Model training summary and evaluation
5. Risk factor explainability
6. Sample claim review output

> **Note:** This notebook walks through the synthetic-fallback pipeline for portability. For real Kaggle dataset results, see `outputs/reports/model_metrics.json` and the **Real Dataset Results** section of `README.md`. The reproducible production pipeline lives in `src/` — run `make real-pipeline`.


In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set path so config.py is importable
sys.path.insert(0, os.path.dirname(os.getcwd()))
import config

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.4f}'.format)
print('Imports OK. BASE_DIR =', config.BASE_DIR)

## 1. Load Data

In [ ]:
raw_path = os.path.join(config.DATA_RAW, 'synthetic_claims.csv')
df = pd.read_csv(raw_path, parse_dates=['claim_date'])
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
df.head(3)

In [ ]:
print('Data Types:')
print(df.dtypes)
print('\nMissing Values:')
print(df.isnull().sum())

## 2. Exploratory Data Analysis

In [ ]:
fraud_rate = df['fraud_label'].mean()
print(f'Overall fraud rate: {fraud_rate:.2%}')
print(f'Fraud count: {df["fraud_label"].sum()} / {len(df)}')
print(f'\nClaim amount statistics:')
print(df[['claim_amount', 'approved_amount']].describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Fraud distribution
df['fraud_label'].value_counts().plot(kind='bar', ax=axes[0], color=['#4CAF50', '#F44336'])
axes[0].set_title('Fraud Label Distribution')
axes[0].set_xticklabels(['Legitimate', 'Fraud'], rotation=0)

# Claim amount by fraud
df[df['fraud_label']==0]['claim_amount'].clip(upper=50000).hist(
    ax=axes[1], bins=40, alpha=0.6, label='Legitimate', color='#4CAF50')
df[df['fraud_label']==1]['claim_amount'].clip(upper=50000).hist(
    ax=axes[1], bins=40, alpha=0.6, label='Fraud', color='#F44336')
axes[1].set_title('Claim Amount by Fraud Label')
axes[1].set_xlabel('Claim Amount ($)')
axes[1].legend()

# Documentation score by fraud
df[df['fraud_label']==0]['documentation_score'].hist(
    ax=axes[2], bins=30, alpha=0.6, label='Legitimate', color='#4CAF50')
df[df['fraud_label']==1]['documentation_score'].hist(
    ax=axes[2], bins=30, alpha=0.6, label='Fraud', color='#F44336')
axes[2].set_title('Documentation Score by Fraud Label')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Fraud rate by service type
fraud_by_service = df.groupby('service_type')['fraud_label'].agg(['mean', 'count'])
fraud_by_service.columns = ['Fraud Rate', 'Total Claims']
fraud_by_service = fraud_by_service.sort_values('Fraud Rate', ascending=False)
print('Fraud Rate by Service Type:')
print(fraud_by_service)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
fraud_by_service['Fraud Rate'].plot(kind='bar', ax=ax, color='#2196F3')
ax.set_title('Fraud Rate by Service Type')
ax.set_ylabel('Fraud Rate')
ax.set_xticklabels(fraud_by_service.index, rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap of key numeric features
numeric_cols = ['claim_amount', 'documentation_score', 'suspicious_keyword_count',
                'duplicate_claim_flag', 'late_submission_flag', 'high_cost_outlier_flag',
                'prior_claim_count', 'provider_claim_volume', 'fraud_label']
fig, ax = plt.subplots(figsize=(10, 8))
corr = df[numeric_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 3. Feature Engineering Overview

In [ ]:
feat_path = os.path.join(config.DATA_PROCESSED, 'claims_features.csv')
if os.path.exists(feat_path):
    df_feat = pd.read_csv(feat_path)
    engineered = ['claim_to_provider_avg_ratio', 'approval_ratio',
                  'claimant_claim_frequency_score', 'provider_risk_score',
                  'documentation_risk_flag', 'suspicious_text_risk_flag',
                  'high_amount_risk_flag', 'rule_based_risk_score']
    existing = [c for c in engineered if c in df_feat.columns]
    print(f'Engineered features available: {existing}')
    print(df_feat[existing].describe().round(3))
else:
    print('Run src/feature_engineering.py first')

In [ ]:
if os.path.exists(feat_path) and 'rule_based_risk_score' in df_feat.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    df_feat[df_feat['fraud_label']==0]['rule_based_risk_score'].hist(
        ax=axes[0], bins=30, alpha=0.6, label='Legitimate', color='#4CAF50')
    df_feat[df_feat['fraud_label']==1]['rule_based_risk_score'].hist(
        ax=axes[0], bins=30, alpha=0.6, label='Fraud', color='#F44336')
    axes[0].set_title('Rule-Based Risk Score Distribution')
    axes[0].legend()
    
    df_feat.boxplot(column='claim_to_provider_avg_ratio', by='fraud_label', ax=axes[1])
    axes[1].set_title('Claim-to-Provider Ratio by Fraud Label')
    axes[1].set_xlabel('Fraud Label (0=Legit, 1=Fraud)')
    plt.tight_layout()
    plt.show()

## 4. Model Training Summary

In [ ]:
metrics_path = os.path.join(config.OUTPUTS_REPORTS, 'model_metrics.json')
if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        metrics = json.load(f)
    metrics_df = pd.DataFrame(metrics).T
    metrics_df = metrics_df.sort_values('roc_auc', ascending=False)
    print('Model Evaluation Metrics (Test Set):')
    print(metrics_df)
else:
    print('Run src/modeling.py first to generate metrics')

In [ ]:
# Display ROC curve
roc_path = os.path.join(config.OUTPUTS_FIGURES, 'roc_curve.png')
if os.path.exists(roc_path):
    from IPython.display import Image
    display(Image(roc_path, width=700))
else:
    print('Run src/modeling.py first')

## 5. Top Risk Factors

In [ ]:
rf_path = os.path.join(config.OUTPUTS_REPORTS, 'top_risk_factors.csv')
if os.path.exists(rf_path):
    rf_df = pd.read_csv(rf_path)
    print('Top 10 Risk Factors:')
    print(rf_df.head(10).to_string(index=False))
else:
    print('Run src/explainability.py first')

In [ ]:
fi_path = os.path.join(config.OUTPUTS_FIGURES, 'feature_importance.png')
if os.path.exists(fi_path):
    from IPython.display import Image
    display(Image(fi_path, width=750))
else:
    print('Run src/modeling.py first')

## 6. Sample Claim Review

In [ ]:
reviews_dir = config.OUTPUTS_REVIEWS
review_files = [f for f in os.listdir(reviews_dir) if f.endswith('.txt')] if os.path.exists(reviews_dir) else []
print(f'Sample reviews available: {len(review_files)}')
if review_files:
    sample_review = os.path.join(reviews_dir, review_files[0])
    with open(sample_review) as f:
        print(f.read())

## 7. Conclusion

This project demonstrates an end-to-end insurance FWA risk scoring pipeline:

- **Data**: 5,000 synthetic claims with realistic fraud patterns (~8% base rate)
- **Feature Engineering**: 8 domain-specific features including ratio-based and flag-based signals
- **Modeling**: Logistic Regression, Random Forest, GradientBoosting/XGBoost, Isolation Forest
- **Explainability**: Feature importance and human-readable claim-level explanations
- **RAG Review**: TF-IDF policy retrieval + template-based structured review reports
- **Dashboard**: Interactive Streamlit app with 5 analytical views

Key findings:
- **Documentation score** and **claim-to-provider-average ratio** are the strongest fraud predictors
- Emergency and Inpatient claims show elevated fraud rates vs Vision/Dental
- Rule-based risk scores correlate strongly with ML model predictions (validates domain logic)
- Best models achieve AUC > 0.85 with balanced recall for fraud detection
